# Set B — Neuron *Structure‑as‑Circuit* (LEGO‑Colab)

**Final (Option 1)** — 2026-03-09  \
**Author:** Satish Nair

> Run the **Starter** once, then execute each section (B1–B10) in order. Short systems‑theoretic callouts and *Try with Copilot* prompts follow each major block. A micro:bit → current‑clamp mapping is provided near the end.

---
## Table of Contents
- [🔰 Colab Starter](#colab-starter)
- [B1. Membrane as Capacitor + Leak as Resistor](#b1)
- [B2. Nernst‑like shifts via `e_pas`](#b2)
- [B3. GHK intuition via leak `g_pas`](#b3)
- [B4. Add HH channels: thresholds](#b4)
- [B5. Pumps proxy via `e_pas` drift](#b5)
- [B6. Time constant (τ): `cm` and `g_pas`](#b6)
- [B7. Length constant (λ): dendrite](#b7)
- [B8. Driving force: `AlphaSynapse` `E_rev`](#b8)
- [B9. Toy myelination & saltatory conduction](#b9)
- [B10. HH threshold search & refractory](#b10)
- [micro:bit → current clamp (CSV pathway)](#microbit)
- [Reflection](#reflection)
---

### 🔰 Colab Starter (run once) <a id='colab-starter'></a>

In [ ]:

# Minimal install and single-compartment setup
!pip -q install neuron==8.2.4
try:
    from neuron import h, gui  # gui harmless in Colab; ok if headless
except Exception:
    from neuron import h
from neuron.units import ms, mV
import matplotlib.pyplot as plt

h.load_file("stdrun.hoc")

# Soma with passive properties
soma = h.Section(name='soma')
soma.L = 20
soma.diam = 20
soma.Ra = 100
soma.insert('pas')
soma.g_pas = 0.0002
soma.e_pas = -65
soma.cm = 1.0

# Recordings
t = h.Vector().record(h._ref_t)
v = h.Vector().record(soma(0.5)._ref_v)

# Utility plotter
def run(sim_ms=50, v_init=-65, title="Membrane Potential"):
    h.finitialize(v_init * mV)
    h.continuerun(sim_ms * ms)
    plt.figure(figsize=(7,3.3))
    plt.plot(t, v, 'k')
    plt.xlabel('Time (ms)')
    plt.ylabel('Voltage (mV)')
    plt.title(title)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

print("Starter ready. Use run(sim_ms=.., v_init=.., title=..) after adding stimuli/mechanisms.")


## B1. Membrane as Capacitor + Leak as Resistor <a id='b1'></a>

In [ ]:

# Passive RC response to a current step
stim = h.IClamp(soma(0.5))
stim.delay = 5
stim.dur   = 50
stim.amp   = -0.05
run(sim_ms=80, title="B1: Passive response to square current (RC-like)")


> **Systems callout — RC membrane**  
- State: V (membrane potential).
- Input: IClamp (current).
- Equilibrium set by leak reversal e_pas and injected current.
- Local dynamics: first‑order τ ≈ C / g (for small perturbations).

**Try with Copilot**  
> Suggest a ± current step that yields ~10 mV peak deflection. Explain how τ shapes the rise/decay.

## B2. Nernst‑like shifts via `e_pas` (exploration) <a id='b2'></a>

In [ ]:

for e_pas in [-80, -70, -60, -50]:
    soma.e_pas = e_pas
    run(sim_ms=60, title=f"B2: Passive cell with e_pas = {e_pas} mV")
soma.e_pas = -65


> **Systems callout — Resting potential as fixed point**  
- Changing e_pas shifts the leak reversal, shifting the V equilibrium.
- Interpretation: proxy for dominant ion gradient / Nernst potential.

**Try with Copilot**  
> Explain why making `e_pas` less negative can depolarize the resting potential without any current injection.

## B3. GHK intuition (vary leak `g_pas`) <a id='b3'></a>

In [ ]:

for g in [0.0001, 0.0002, 0.0005, 0.001]:
    soma.g_pas = g
    run(sim_ms=60, title=f"B3: Effect of leak (permeability proxy), g_pas={g} S/cm²")
soma.g_pas = 0.0002


> **Systems callout — Conductance as gain & τ**  
- Increasing g_pas decreases τ and reduces voltage excursion for a given I.
- Think: stronger leak clamps V closer to e_pas (lower DC gain). 

**Try with Copilot**  
> For a fixed current step, predict how the peak deflection scales when `g_pas` doubles. Justify mathematically.

## B4. Voltage‑gated channels — add HH <a id='b4'></a>

In [ ]:

soma.insert('hh')
stim.amp = 0.03
run(sim_ms=60, title="B4: HH present, subthreshold current")
stim.amp = 0.08
run(sim_ms=60, title="B4: HH present, suprathreshold current (spike expected)")


> **Systems callout — Nonlinear threshold**  
- State now includes gating variables (m, h, n) implicitly in mechanism.
- Input: IClamp. Threshold emerges from positive feedback in Na⁺ activation vs K⁺ activation.

**Try with Copilot**  
> Propose an amplitude just below threshold and one just above. Explain using the relative timing of Na⁺ activation and K⁺ activation.

## B5. Pumps as rechargers (qualitative drift via `e_pas`) <a id='b5'></a>

In [ ]:

for e_shift in [-65, -60, -55]:
    soma.e_pas = e_shift
    stim.amp = 0.0
    run(sim_ms=200, title=f"B5: Drift with altered e_pas={e_shift} mV (pump proxy)")
soma.e_pas = -65
stim.amp = 0.08


> **Systems callout — Slow drift as baseline modulation**  
- Interpreting e_pas shifts as long‑term homeostatic/pump effects.
- Design thought: what measurements would reveal such baseline drifts?

**Try with Copilot**  
> Describe an experiment to detect slow baseline depolarization and distinguish it from input‑driven responses.

## B6. Time constant (τ) — change `cm` and leak <a id='b6'></a>

In [ ]:

stim.amp  = -0.05
stim.dur  = 40
stim.delay= 5
configs = [
    ("cm=1.0, g_pas=0.0002", 1.0, 0.0002),
    ("cm=2.0, g_pas=0.0002", 2.0, 0.0002),
    ("cm=1.0, g_pas=0.0005", 1.0, 0.0005),
]
for label, cm, gpas in configs:
    soma.cm = cm
    soma.g_pas = gpas
    run(sim_ms=70, title=f"B6: τ effect — {label}")
soma.cm=1.0; soma.g_pas=0.0002


> **Systems callout — τ ≈ C_total / g_total**  
- Doubling cm roughly doubles τ (slower dynamics).
- Increasing g_pas reduces τ (faster return to baseline). 

**Try with Copilot**  
> Estimate τ from the 63% rise time in one of your traces; compare with C/g.

## B7. Length constant (λ) — add a dendrite <a id='b7'></a>

In [ ]:

dend = h.Section(name='dend')
dend.L = 500; dend.diam = 1.5; dend.Ra = 100
dend.insert('pas')
dend.g_pas = soma.g_pas; dend.e_pas = soma.e_pas
dend.connect(soma(1.0))

v_soma = h.Vector().record(soma(0.5)._ref_v)
v_tip  = h.Vector().record(dend(1.0)._ref_v)

stim_d = h.IClamp(dend(1.0))
stim_d.delay = 5; stim_d.dur = 40; stim_d.amp = -0.05

h.finitialize(-65*mV); h.continuerun(70*ms)
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3.3))
plt.plot(t, v_soma, label='Soma(0.5)')
plt.plot(t, v_tip,  label='Dend tip(1.0)')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('B7: Dendritic attenuation')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

# Thicker dendrite
dend.diam = 3.0
h.finitialize(-65*mV); h.continuerun(70*ms)
plt.figure(figsize=(7,3.3))
plt.plot(t, v_soma, label='Soma(0.5)')
plt.plot(t, v_tip,  label='Dend tip(1.0) — thicker')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('B7: Bigger diameter, less attenuation')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Spatial filtering**  
- λ increases with diameter and membrane resistance; larger λ → less attenuation.
- Design: where to inject/record to infer λ from attenuation curves?

**Try with Copilot**  
> Predict how doubling dendrite diameter affects attenuation and why (think axial vs membrane resistances).

## B8. Driving force — `AlphaSynapse` with different `E_rev` <a id='b8'></a>

In [ ]:

syn = h.AlphaSynapse(soma(0.5))
syn.onset = 10; syn.tau = 2; syn.gmax = 0.001
for erev in [0, -70, -55]:
    syn.e = erev
    h.finitialize(-65*mV); h.continuerun(40*ms)
    plt.figure(figsize=(7,3.3))
    plt.plot(t, v, 'k')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)')
    plt.title(f"B8: AlphaSynapse with E_rev={erev} mV")
    plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Driving force & sign**  
- If V < E_rev (excitatory), current is inward → depolarization; if V > E_rev, outward → hyperpolarization.
- Maps to synapse identity: glutamatergic‑like (≈0 mV) vs GABAergic‑like (≈−70 mV). 

**Try with Copilot**  
> For `E_rev = −70 mV`, explain why synaptic activation near rest produces little voltage change compared to `E_rev = 0 mV`.

## B9. Myelination — toy multi‑section axon <a id='b9'></a>

In [ ]:

axon = [h.Section(name=f'ax_{i}') for i in range(5)]
for i,sec in enumerate(axon):
    sec.L = 100; sec.diam = 1.0; sec.Ra = 100; sec.insert('pas')
    if i>0:
        sec.connect(axon[i-1](1.0))

node_idx = [0,2,4]
for i,sec in enumerate(axon):
    if i in node_idx:
        sec.cm = 1.0;  sec.g_pas = 0.001
    else:
        sec.cm = 0.02; sec.g_pas = 0.00002

stim_ax = h.IClamp(axon[0](0.5)); stim_ax.delay=1; stim_ax.dur=1; stim_ax.amp=1.0
recs = [h.Vector().record(sec(0.5)._ref_v) for sec in axon]
t_ax = h.Vector().record(h._ref_t)

h.finitialize(-65*mV); h.continuerun(5*ms)
plt.figure(figsize=(7,3.3))
for i,vec in enumerate(recs):
    plt.plot(t_ax, vec, label=f'ax_{i}')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('B9: Toy saltatory conduction (qualitative)')
plt.legend(); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Conduction economy**  
- Myelin ↓ cm and ↓ g_pas, allowing faster, farther spread between nodes.
- Nodes act as active boosters; internodes as low‑leak cables.

**Try with Copilot**  
> Describe how increasing internode length might speed conduction up to a point, then hurt reliability.

## B10. HH — threshold and refractory <a id='b10'></a>

In [ ]:

soma.insert('hh')
amps = [0.02,0.04,0.06,0.08,0.10]
for A in amps:
    stim.delay=5; stim.dur=10; stim.amp=A
    run(sim_ms=30, title=f"B10: Step current A={A} nA (threshold search)")

stim1 = h.IClamp(soma(0.5)); stim1.dur=2; stim1.amp=0.15
stim2 = h.IClamp(soma(0.5)); stim2.dur=2; stim2.amp=0.15
for isi in [2,4,6,8,10]:
    stim1.delay=5
    stim2.delay=5+isi
    run(sim_ms=25, title=f"B10: Paired pulses, ISI={isi} ms")


> **Systems callout — Refractory timing**  
- Absolute refractory from Na⁺ inactivation; relative refractory from residual K⁺ activation.
- ISI sweep reveals recovery curve; design: measure spike probability vs ISI.

**Try with Copilot**  
> Estimate the relative refractory period from your paired‑pulse series and explain the ion‑channel basis.

## micro:bit → current clamp (CSV pathway) <a id='microbit'></a>
Default mapping: **accelerometer magnitude → IClamp current**.

In [ ]:

# Upload a CSV from micro:bit logs (or any sensor CSV)
# Expected columns (flexible): either
#   1) time_ms, accel  (accel arbitrary units)
# or 2) accel only, in which case time is inferred as uniform dt

import io, csv
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()  # pick one file
name = list(uploaded.keys())[0]
raw = uploaded[name].decode('utf-8')
reader = csv.reader(io.StringIO(raw))
rows = [r for r in reader if len(r)>0]

# Try to detect header
header = None
try:
    _ = [float(x) for x in rows[0]]
except Exception:
    header = rows[0]
    rows = rows[1:]

# Parse numeric
data = np.array([[float(x) for x in r[:2]] if len(r)>=2 else [np.nan, float(r[0])] for r in rows])

if np.isnan(data[:,0]).all():
    # no time provided → assume dt=10 ms
    dt_ms = 10.0
    t_ms = np.arange(len(data)) * dt_ms
    accel = data[:,1]
else:
    t_ms = data[:,0]
    accel = data[:,1]

# Preprocess accelerometer magnitude → current (nA)
# Center and scale; tweak k and baseline as needed.
baseline = np.percentile(accel, 10)
k = 0.02  # nA per accel unit (adjust to taste)
I_nA = k * (accel - baseline)
I_nA[I_nA < 0] = 0.0  # rectification optional

plt.figure(figsize=(7,3))
plt.plot(t_ms, accel, label='accel (a.u.)', color='tab:gray')
plt.legend(); plt.title('Uploaded accelerometer (a.u.)'); plt.xlabel('time (ms)'); plt.ylabel('a.u.')
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,3))
plt.plot(t_ms, I_nA, label='I (nA) mapped from accel', color='tab:blue')
plt.legend(); plt.title('Mapped current clamp (nA)'); plt.xlabel('time (ms)'); plt.ylabel('nA')
plt.tight_layout(); plt.show()

# Create a piecewise current injection using a Vector play
stim_mb = h.IClamp(soma(0.5))
stim_mb.delay = 0
stim_mb.dur = max(1.0, float(t_ms[-1]-t_ms[0]))  # ms

# Interpolate to 0.1 ms grid for smooth play
grid_dt = 0.1
grid_t = np.arange(t_ms[0], t_ms[-1]+grid_dt, grid_dt)
I_interp = np.interp(grid_t, t_ms, I_nA)

# Build play vectors
vec_t = h.Vector(grid_t)
vec_i = h.Vector(I_interp)
vec_i.play(stim_mb._ref_amp, vec_t, 1)  # 1 ms reference, times in ms

h.finitialize(-65*mV)
h.continuerun(float(grid_t[-1]))

plt.figure(figsize=(7,3.3))
plt.plot(t, v, 'k')
plt.title('micro:bit‑driven membrane response')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)')
plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


**Notes**  
- Adjust `k` and `baseline` to get usable currents.  
- Remove rectification if you want bidirectional drive.  
- Swap mapping to a synaptic conductance by replacing `IClamp` with `AlphaSynapse`/`ExpSyn` and playing **`gmax`** instead of **`amp`**.

## Reflection <a id='reflection'></a>
- Which parameters most strongly affect threshold in B4/B10?  
- How do τ (B6) and λ (B7) connect to signal filtering and spatial attenuation?  
- Where would you place a micro:bit‑derived input (current vs synaptic conductance) to emulate a sensor pathway?